In [ ]:
import pandas as pd

from testing.r_and_d.boolean_queries.query_tester import (
    BooleanQueryTester,
    use_baseline_query,
    generate_with_llm,
    query_openalex_minimal,
    reference_df,
)

from testing.r_and_d.boolean_queries.prompts import (
    policy_atlas_v1,
    policy_atlas_v2,
    wang_et_al_q2_prompt,
    wang_et_al_q3_prompt,
)

from app.services.openalex import sanitize_openalex_query
from testing import TESTING_DIR, logger

BOOL_DIR = TESTING_DIR / "r_and_d/boolean_queries/outputs/"
baseline_results = pd.read_json(BOOL_DIR / "baseline_results.jsonl", lines=True)


In [ ]:
q = reference_df.question.iloc[3]
q

In [ ]:
query = await use_baseline_query(q, "model", 0, "prompt")
query = sanitize_openalex_query(query)
query

In [ ]:
await query_openalex_minimal(query, count_only=True)

## Prompting experiments

In [ ]:
test_prompt = """
Transform user input into a high quality boolean query 
for querying the OpenAlex academic research database.

# Guidance from OpenAlex
Imagine you are an expert systematic review information specialist; now you are given a systematic review
research topic, with the topic title provided by the user below. Your task is to generate a highly effective systematic
review Boolean query to search on OpenAlex (refer to the professionally made ones); the query needs to be
as inclusive as possible so that it can retrieve all the relevant studies that can be included in the research
topic; on the other hand, the query needs to retrieve fewer irrelevant studies so that researchers can spend
less time judging the retrieved documents.

# Important instructions

Return ONLY the boolean query string, nothing else.
"""

In [ ]:
query = await generate_with_llm(
    question=q,
    model="gpt-5",
    temperature=1.0,
    system_prompt=wang_et_al_q3_prompt,
)
logger.info(q)
logger.info("=" * 20)
logger.info(query)
logger.info("=" * 20)
n_results = await query_openalex_minimal(query, count_only=True)
logger.info(f"Number of retrieved results: {n_results}")

In [ ]:
async def repeated_queries(
    question: str,
    model: str,
    temperature: float,
    system_prompt: str,
    n_runs: int = 5,
) -> str:
    """Generate a boolean query using the given parameters."""
    dfs = []
    for _ in range(n_runs):
        query = await generate_with_llm(
            question=question,
            model=model,
            temperature=temperature,
            system_prompt=system_prompt,
        )
        df, _ = await query_openalex_minimal(query, count_only=False)
        dfs.append(df.assign(query=query))
    return (
        pd.concat(dfs)
        .drop_duplicates(subset=["id"])
    )
    

In [ ]:
q = "what is the impact of financial incentive on parental engagement in evidence based parenting programmes?"

In [ ]:
results = await repeated_queries(q, "gpt-5-mini", 1.0, wang_et_al_q3_prompt, n_runs=3)
len(results)

In [ ]:
results

In [ ]:
pd.set_option("display.max_colwidth", 200)
results.sort_values("relevance_score", ascending=False)[["title", "relevance_score", "query"]]